In [ ]:
!pip install py-vncorenlp peft bitsandbytes

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, MBartModel, AutoConfig
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
import py_vncorenlp  
from torch.cuda.amp import autocast, GradScaler

class Config:
    TRAIN_PATH = '/content/neu/train_set.csv'
    VAL_PATH = '/content/neu/val_set.csv'

    MODEL_NAME = "vinai/bartpho-word"
    MAX_LEN = 512

    BATCH_SIZE = 16
    ACCUMULATION_STEPS = 2  

    EPOCHS = 8
    LEARNING_RATE = 2e-4

    LORA_R = 16             
    LORA_ALPHA = 32         
    LORA_DROPOUT = 0.05

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    RDR_PATH = '/content/output'


In [ ]:
def setup_vncorenlp():
    should_download = True
    if os.path.exists(Config.RDR_PATH):
        if os.path.exists(os.path.join(Config.RDR_PATH, 'VnCoreNLP-1.2.jar')):
            should_download = False
        else:
            shutil.rmtree(Config.RDR_PATH)

    if should_download:
        print("Đang tải VnCoreNLP (v1.2)...")
        try:
            os.makedirs(Config.RDR_PATH, exist_ok=True)
            py_vncorenlp.download_model(save_dir=Config.RDR_PATH)
            print("Tải thành công!")
        except Exception as e:
            print(f"Lỗi tải VnCoreNLP: {e}")

setup_vncorenlp()

# Khởi tạo RDRSegmenter toàn cục
try:
    if 'rdrsegmenter' not in globals():
        rdrsegmenter = py_vncorenlp.VnCoreNLP(save_dir=Config.RDR_PATH, annotators=["wseg"])
        print("✅ RDRSegmenter đã sẵn sàng.")
except Exception as e:
    print(f"⚠️ Lỗi khởi tạo Java: {e}")

def segment_text(text):
    if not isinstance(text, str) or not text.strip(): return ""
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join(sentences)
    except:
        return text

def get_class_weights(y, device):
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
    return torch.tensor(class_weights, dtype=torch.float).to(device)

class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = df['text'].values
        self.sentiments = df['sentiment'].values
        self.topics = df['classification'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.texts[index])
        text = raw_text
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(self.sentiments[index], dtype=torch.long),
            'topic_label': torch.tensor(self.topics[index], dtype=torch.long)
        }

from peft import get_peft_model, LoraConfig, TaskType

class MultiTaskBARTpho(nn.Module):
    def __init__(self, n_sentiment, n_topic):
        super(MultiTaskBARTpho, self).__init__()
        print(f"Loading {Config.MODEL_NAME} with LoRA configuration...")

        # 1. Load Base Model
        self.bart = MBartModel.from_pretrained(Config.MODEL_NAME)

        # 2. Config LoRA
        # target_modules=["q_proj", "v_proj"] là quan trọng nhất cho Transformer
        peft_config = LoraConfig(
            inference_mode=False,
            r=Config.LORA_R,
            lora_alpha=Config.LORA_ALPHA,
            lora_dropout=Config.LORA_DROPOUT,
            bias="none",
            target_modules=["q_proj", "v_proj"]
        )

        # 3. Inject LoRA (Tự động đóng băng các layer gốc)
        self.bart = get_peft_model(self.bart, peft_config)
        self.bart.print_trainable_parameters()

        # 4. Custom Heads (Sẽ được train cùng với LoRA adapters)
        hidden_size = self.bart.config.hidden_size
        self.drop = nn.Dropout(p=0.1)

        self.sent_out = nn.Linear(hidden_size, n_sentiment)
        self.topic_out = nn.Linear(hidden_size, n_topic)

    def forward(self, input_ids, attention_mask):
        # Base model (wrapped by LoRA)
        encoder_outputs = self.bart(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = encoder_outputs.last_hidden_state

        # Lấy token đầu tiên làm đại diện
        cls_output = last_hidden_state[:, 0, :]
        output = self.drop(cls_output)

        return self.sent_out(output), self.topic_out(output)

def train_epoch(model, dataloader, optimizer, device, w_sent=None, w_topic=None):
    model.train()
    total_loss = 0

    crit_sent = nn.CrossEntropyLoss(weight=w_sent)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic)
    scaler = GradScaler()
    optimizer.zero_grad()

    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        sent_lbl = batch['sentiment_label'].to(device)
        topic_lbl = batch['topic_label'].to(device)

        with autocast():
            sent_logits, topic_logits = model(input_ids, mask)
            loss_s = crit_sent(sent_logits, sent_lbl)
            loss_t = crit_topic(topic_logits, topic_lbl)
            loss = (loss_s + loss_t) / Config.ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % Config.ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        current_loss = loss.item() * Config.ACCUMULATION_STEPS
        total_loss += current_loss

        del input_ids, mask, sent_lbl, topic_lbl, sent_logits, topic_logits, loss

        progress_bar.set_postfix({'loss': f"{current_loss:.4f}"})

    return total_loss / len(dataloader)

def validate_epoch(model, dataloader, device, w_sent=None, w_topic=None):
    model.eval()
    total_loss = 0
    correct_sent = 0
    correct_topic = 0
    total_samples = 0

    crit_sent = nn.CrossEntropyLoss(weight=w_sent)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            sent_lbl = batch['sentiment_label'].to(device)
            topic_lbl = batch['topic_label'].to(device)

            sent_logits, topic_logits = model(input_ids, mask)

            loss = crit_sent(sent_logits, sent_lbl) + crit_topic(topic_logits, topic_lbl)
            total_loss += loss.item()

            pred_sent = torch.argmax(sent_logits, dim=1)
            correct_sent += (pred_sent == sent_lbl).sum().item()

            pred_topic = torch.argmax(topic_logits, dim=1)
            correct_topic += (pred_topic == topic_lbl).sum().item()

            total_samples += sent_lbl.size(0)

    return total_loss / len(dataloader), correct_sent / total_samples, correct_topic / total_samples


In [ ]:
if __name__ == '__main__':

    print("--- 1. Loading Data ---")
    try:
        df_train = pd.read_csv(Config.TRAIN_PATH)
        df_val = pd.read_csv(Config.VAL_PATH)
        print(f"Train size: {len(df_train)} | Val size: {len(df_val)}")
    except FileNotFoundError:
        print("❌ Không tìm thấy file (Running Dummy Mode).")
        df_train = pd.DataFrame({'text': ['Demo text'], 'sentiment': [0], 'classification': [0]})
        df_val = df_train.copy()

    print("Đang tách từ (Segmentation)... có thể mất 1-2 phút...")

    df_train['text'] = df_train['text'].apply(segment_text)
    df_val['text'] = df_val['text'].apply(segment_text)


    print("✅ Tách từ hoàn tất!")

    n_sent = int(df_train['sentiment'].max()) + 1
    n_topic = int(df_train['classification'].max()) + 1

    print("\n--- 2. Preparing Dataset ---")
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

    train_ds = MultiTaskDataset(df_train, tokenizer, Config.MAX_LEN)
    val_ds = MultiTaskDataset(df_val, tokenizer, Config.MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, num_workers=2)

    w_sent = get_class_weights(df_train['sentiment'].values, Config.DEVICE)
    w_topic = get_class_weights(df_train['classification'].values, Config.DEVICE)

    print("--- 3. Initializing BARTpho with LoRA ---")
    model = MultiTaskBARTpho(n_sent, n_topic)
    model.to(Config.DEVICE)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=Config.LEARNING_RATE)

    best_val_loss = float('inf')

    print("--- 4. Start Training ---")
    for epoch in range(Config.EPOCHS):
        print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")

        train_loss = train_epoch(model, train_loader, optimizer, Config.DEVICE, w_sent, w_topic)
        val_loss, val_acc_sent, val_acc_topic = validate_epoch(model, val_loader, Config.DEVICE, w_sent, w_topic)

        print(f" -> Train Loss: {train_loss:.4f}")
        print(f" -> Val Loss  : {val_loss:.4f} | Sent Acc: {val_acc_sent:.4f} | Topic Acc: {val_acc_topic:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "bartpho_lora_best.pth")
            print(" ✅ Saved Best Model")

    print("\n✅ Training Completed!")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

ID2SENTIMENT = {
    0: "Neutral", 1: "Positive", 2: "Negative", 3: "Toxic"
}
ID2TOPIC = {
    0: "Spam", 1: "News", 2: "Academic", 3: "Other", 4: "Service",
    5: "Jobs & Recruitment", 6: "Personal Affairs", 7: "Social Affairs",
    8: "Help & Share", 9: "Club & Events"
}
# Đảo ngược để dùng cho report
s_names = [ID2SENTIMENT[i] for i in sorted(ID2SENTIMENT.keys())]
t_names = [ID2TOPIC[i] for i in sorted(ID2TOPIC.keys())]

def evaluate_test_set(model, tokenizer, device):
    print("\n--- 5. Loading Test Data & Evaluating ---")
    try:
        TEST_PATH = '/content/neu/test_set.csv'
        df_test = pd.read_csv(TEST_PATH)
        print(f"Test size: {len(df_test)}")
    except Exception as e:
        print(f"❌ Lỗi load file test: {e}. Đang dùng Dummy Data để test code.")
        df_test = pd.DataFrame({
            'text': ['Test sentence'], 'sentiment': [1], 'classification': [2]
        })

    test_ds = MultiTaskDataset(df_test, tokenizer, Config.MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False)

    model.eval()
    y_true_s, y_pred_s = [], []
    y_true_t, y_pred_t = [], []

    print("Running Prediction on Test Set...")
    with torch.no_grad():
        for batch in tqdm(test_loader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)

            # Forward
            logits_s, logits_t = model(input_ids, mask)

            # Get Predictions
            preds_s = torch.argmax(logits_s, dim=1).cpu().numpy()
            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()

            y_true_s.extend(batch['sentiment_label'].cpu().numpy())
            y_pred_s.extend(preds_s)
            y_true_t.extend(batch['topic_label'].cpu().numpy())
            y_pred_t.extend(preds_t)

    # Report Sentiment
    print("\n" + "="*40)
    print("KẾT QUẢ: CẢM XÚC (SENTIMENT)")
    print("="*40)
    print(f"Accuracy: {accuracy_score(y_true_s, y_pred_s):.4f}\n")
    print(classification_report(y_true_s, y_pred_s, target_names=s_names, zero_division=0))

    # Report Topic
    print("\n" + "="*40)
    print("KẾT QUẢ: CHỦ ĐỀ (TOPIC)")
    print("="*40)
    print(f"Accuracy: {accuracy_score(y_true_t, y_pred_t):.4f}\n")
    print(classification_report(y_true_t, y_pred_t, target_names=t_names, zero_division=0))

# Chạy đánh giá
if __name__ == '__main__':
    # Nếu biến model đã train xong ở bước trước
    evaluate_test_set(model, tokenizer, Config.DEVICE)

    # --- C. LƯU MODEL FINAL (QUAN TRỌNG) ---
    print("\n--- 6. Saving Final Model ---")

    # LƯU Ý QUAN TRỌNG VỀ LORA VÀ CUSTOM HEADS:
    # Model của chúng ta gồm:
    #   1. BARTpho base (đã đóng băng)
    #   2. LoRA Adapters (đã train)
    #   3. Hai lớp Linear Custom (sent_out, topic_out - đã train)
    # Nếu chỉ dùng model.bart.save_pretrained(), nó sẽ MẤT 2 lớp Linear.
    # -> Cách an toàn nhất: Lưu toàn bộ state_dict của wrapper class.

    torch.save(model.state_dict(), "bartpho_lora_multitask_complete.pth")
    tokenizer.save_pretrained("./bartpho_tokenizer_final")

    print("✅ Đã lưu model tại: bartpho_lora_multitask_complete.pth")
    print("✅ Đã lưu tokenizer tại: ./bartpho_tokenizer_final")